# Estudio 02 — Arquitecturas de Deep Learning: CNN y RNN
**Diplomado Superior en Redes Neuronales Artificiales y Deep Learning**  
Módulo 4 | Diana Blanco

---
## 🖼️ CNN — Redes Neuronales Convolucionales

### ¿Qué es una convolución?
Una convolución aplica un **kernel (filtro)** que desliza sobre la imagen para extraer características como bordes, texturas y formas.

### Componentes clave:
- **Capa Convolucional:** Aplica $N$ filtros que aprenden patrones visuales
- **Pooling (Max/Average):** Reduce dimensionalidad espacial, mantiene invarianza
- **Feature Maps:** Mapas de activación después de cada filtro
- **GlobalAveragePooling:** Reduce cada feature map a un solo valor antes de la clasificación

```
Imagen → [Conv2D + ReLU] → [MaxPooling] → ... → [GlobalAvgPool] → [Dense] → Salida
```

---
## 🔄 RNN / LSTM — Redes Recurrentes

### ¿Qué es una red recurrente?
Mantiene un **estado oculto** que pasa información entre pasos de tiempo, ideal para secuencias.

### LSTM (Long Short-Term Memory)
Resuelve el problema de vanishing gradient con **compuertas (gates)**:
- **Forget gate:** Decide qué información del estado anterior descartar
- **Input gate:** Decide qué nueva información almacenar
- **Output gate:** Decide qué información pasar al siguiente paso

```
Texto → [Embedding] → [Bidirectional LSTM] → [Dense] → Salida
```

In [ ]:
# MACHOTE: Configuración inicial
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from modulo4_libreria import *

INFO = setup_completo()

In [ ]:
# MACHOTE: Cargar MNIST desde keras.datasets
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"Train: {X_train.shape}, labels: {y_train.shape}")
print(f"Test:  {X_test.shape},  labels: {y_test.shape}")

# Normalizar y redimensionar para CNN (28,28,1)
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# Dividir train para obtener validación
datos = train_val_test_split(X_train, y_train, test_size=0.0, val_size=0.1)
X_val, y_val = datos['X_val'], datos['y_val']
del datos

In [ ]:
# MACHOTE: Construir CNN para MNIST (10 dígitos)
cnn_mnist = crear_cnn(
    input_shape=(28, 28, 1),
    num_clases=10,
    filtros=[32, 64],
    kernel_size=3
)

cnn_mnist.summary()

In [ ]:
# MACHOTE: Entrenar CNN en MNIST
hist_cnn = compilar_y_entrenar(
    cnn_mnist, X_train, y_train, X_val, y_val,
    num_clases=10, lr=0.001, epochs=10, batch_size=128,
    early_stop_paciencia=3
)

graficar_historia(hist_cnn, titulo="CNN - MNIST")

In [ ]:
# MACHOTE: Cargar IMDB para clasificación de texto con LSTM
from tensorflow.keras.datasets import imdb

vocab_size = 10000
max_len = 200

(X_train_text, y_train_text), (X_test_text, y_test_text) = imdb.load_data(
    num_words=vocab_size
)

print(f"Train: {len(X_train_text)} secuencias")
print(f"Test:  {len(X_test_text)} secuencias")

# Padding a longitud fija
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_train_pad = pad_sequences(X_train_text, maxlen=max_len, padding='pre', truncating='pre')
X_test_pad  = pad_sequences(X_test_text, maxlen=max_len, padding='pre', truncating='pre')

# Separar validación
datos_text = train_val_test_split(X_train_pad, y_train_text, test_size=0.0, val_size=0.1)
X_val_pad, y_val_text = datos_text['X_val'], datos_text['y_val']
del datos_text

In [ ]:
# MACHOTE: Construir LSTM para clasificación de texto
lstm_imdb = crear_lstm_texto(
    vocab_size=vocab_size,
    max_len=max_len,
    embedding_dim=64,
    num_clases=2
)

lstm_imdb.summary()

In [ ]:
# MACHOTE: Entrenar LSTM en IMDB
hist_lstm = compilar_y_entrenar(
    lstm_imdb, X_train_pad, y_train_text, X_val_pad, y_val_text,
    num_clases=2, lr=0.001, epochs=5, batch_size=64,
    early_stop_paciencia=3
)

graficar_historia(hist_lstm, titulo="LSTM - IMDB")

---
## 📊 Comparación: CNN vs RNN/LSTM

| Aspecto | CNN | RNN / LSTM |
|---|---|---|
| **Tipo de datos** | Imágenes, grids | Secuencias, texto, series tiempo |
| **Operación clave** | Convolución + Pooling | Estado oculto recurrente |
| **Paralelización** | Alta (independiente por pixel) | Baja (secuencial) |
| **Ventaja** | Detecta patrones espaciales | Modela dependencias temporales |
| **Problema típico** | Perder contexto global | Vanishing gradient (LSTM lo mitiga) |
| **Uso común** | Clasificación de imágenes, detección | Sentiment analysis, traducción |

---
## ✏️ Ejercicios (Tarea)

1. Cambia `filtros=[64, 128, 256]` en la CNN. ¿Cómo cambia el número de parámetros?
2. Prueba CIFAR-10 en lugar de MNIST: `keras.datasets.cifar10.load_data()`. ¿Qué cambia en `input_shape`?
3. En el LSTM, cambia `embedding_dim` a 128. ¿Mejora el accuracy?
4. Prueba con `crear_lstm_texto` usando sólo 1 capa LSTM (sin bidireccional). ¿Cómo cambia?

In [ ]:
# TODO: Experimenta con diferentes arquitecturas
# 1. Carga CIFAR-10 y construye una CNN con input_shape=(32,32,3)
# 2. Crea un LSTM unidireccional simple (sin Bidirectional)
# 3. Compara tiempos de entrenamiento

# === ESCRIBE TU CÓDIGO AQUÍ ===



# ════════════════════════════════════